In [87]:
import pandas as pd
from math import * 
import json 
import random 
import hashlib
import numpy as np

import json
from pathlib import Path
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

In [80]:
RANDOM_SEED = 20260901

TRAIN_SIZE = 0.80
VALIDATION_SIZE = 0.20

In [81]:
from pathlib import Path

print("Current folder:")
print(Path.cwd())

print("\nFiles/folders here:")
for item in Path.cwd().iterdir():
    print(item.name)

Current folder:
c:\Users\m_laibhen2024\Downloads\Accenture_1D\Accenture-1D-contract-review-challenge-1\notebooks

Files/folders here:
.gitkeep
create_split.ipynb
__pycache__


In [101]:
# Project folder
import json

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "cuad" / "train_separate_questions.json"

print("Training-approved data: ")
print(DATA_PATH)
print("File exists:", DATA_PATH.exists())
# Find project root

# Path to original CUAD file

# Folder where the split files will be saved
OUTPUT_DIR = PROJECT_ROOT / "data" / "splits"

# Create the folder if it does not exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Paths for the new files
TRAIN_PATH = OUTPUT_DIR / "train.json"
VALIDATION_PATH = OUTPUT_DIR / "validation.json"
MANIFEST_PATH = OUTPUT_DIR / "split_manifest.csv"


with open(DATA_PATH, "r", encoding="utf-8") as file:
    data = json.load(file)
contracts = data["data"]

print("Contracts:", len(contracts))

assert len(contracts) == 408, (
    f"Expected 408 training-approved contracts, "
    f"found {len(contracts)}"
)




Training-approved data: 
c:\Users\m_laibhen2024\Downloads\Accenture_1D\Accenture-1D-contract-review-challenge-1\data\cuad\train_separate_questions.json
File exists: True
Contracts: 408


In [83]:
#looking through the data and understanidng it 
print("\nTop-level keys:")
print(data.keys())

first_contract = contracts[0]

print("\nContract keys:")
print(first_contract.keys())

print("\nTitle:")
print(first_contract["title"])

print("\nParagraph count:")
print(len(first_contract["paragraphs"]))

first_paragraph = first_contract["paragraphs"][0]

print("\nParagraph keys:")
print(first_paragraph.keys())

first_qa = first_paragraph["qas"][0]

print("\nQA keys:")
print(first_qa.keys())

print("\nExample QA:")
print(first_qa)


Top-level keys:
dict_keys(['version', 'data'])

Contract keys:
dict_keys(['title', 'paragraphs'])

Title:
LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT

Paragraph count:
1

Paragraph keys:
dict_keys(['qas', 'context'])

QA keys:
dict_keys(['answers', 'id', 'question', 'is_impossible'])

Example QA:
{'answers': [{'text': 'DISTRIBUTOR AGREEMENT', 'answer_start': 44}], 'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Document Name_0', 'question': 'Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract', 'is_impossible': False}


In [84]:

num_contracts = len(contracts)

num_documents = sum(
    len(contract["paragraphs"])
    for contract in contracts
)


# Count unique Contract × Category combinations
annotation_sets = set()

for contract_index, contract in enumerate(contracts):
    for paragraph in contract["paragraphs"]:
        for qa in paragraph["qas"]:
            
            annotation_sets.add(
                (
                    contract_index,
                    qa["question"]
                )
            )

num_annotation_sets = len(annotation_sets)

print("Annotation Sets:", num_annotation_sets)
num_spans = sum(
    len(qa["answers"])
    for contract in contracts
    for paragraph in contract["paragraphs"]
    for qa in paragraph["qas"]
)

print("\n--- DATASET TOTALS ---")
print("Contracts:", num_contracts)
print("Documents:", num_documents)
print("Annotation Sets:", num_annotation_sets)
print("Spans:", num_spans)


Annotation Sets: 16728

--- DATASET TOTALS ---
Contracts: 408
Documents: 408
Annotation Sets: 16728
Spans: 11180


In [107]:
assert num_contracts == 408, (
    f"Expected 408 contracts, found {num_contracts}"
)

assert num_documents == 408, (
    f"Expected 408 documents, found {num_documents}"
)

assert num_annotation_sets == 16728, (
    f"Expected 16,728 annotation sets, found {num_annotation_sets}"
)

assert num_spans == 11180, (
    f"Expected 11,180 spans, found {num_spans}"
)

In [ ]:
# verify split size 

questions = []

for contract in contracts:
    for paragraph in contract["paragraphs"]:
        for qa in paragraph["qas"]:
            questions.append(qa["question"])

unique_questions = sorted(set(questions))

print("Unique category questions:", len(unique_questions))

for i, question in enumerate(unique_questions, start=1):
    print(i, question)
    
assert len(unique_questions) == 41, (
    f"Expected 41 categories, found {len(unique_questions)}"
)

categories = unique_questions

Unique category questions: 41
1 Highlight the parts (if any) of this contract related to "Affiliate License-Licensee" that should be reviewed by a lawyer. Details: Does the contract contain a license grant to a licensee (incl. sublicensor) and the affiliates of such licensee/sublicensor?
2 Highlight the parts (if any) of this contract related to "Affiliate License-Licensor" that should be reviewed by a lawyer. Details: Does the contract contain a license grant by affiliates of the licensor or that includes intellectual property of affiliates of the licensor? 
3 Highlight the parts (if any) of this contract related to "Agreement Date" that should be reviewed by a lawyer. Details: The date of the contract
4 Highlight the parts (if any) of this contract related to "Anti-Assignment" that should be reviewed by a lawyer. Details: Is consent or notice required of a party if the contract is assigned to a third party?
5 Highlight the parts (if any) of this contract related to "Audit Rights" tha

In [ ]:
annotation_set_errors = []

for contract in contracts:

    qas = [
        qa
        for paragraph in contract["paragraphs"]
        for qa in paragraph["qas"]
    ]

    if len(qas) != 41:
        annotation_set_errors.append({
            "title": contract["title"],
            "annotation_sets": len(qas)
        })

print(
    "Contracts without exactly 41 annotation sets:",
    len(annotation_set_errors)
)

assert len(annotation_set_errors) == 0, (
    "Some contracts do not contain exactly 41 annotation sets."
)

Contracts without exactly 41 annotation sets: 407


AssertionError: Some contracts do not contain exactly 41 annotation sets.

In [ ]:
titles = [
    contract["title"]
    for contract in contracts
]

print("\n--- TITLE CHECK ---")
print("Total titles:", len(titles))
print("Unique titles:", len(set(titles)))
print(
    "Duplicate titles:",
    len(titles) - len(set(titles))
)
assert len(titles) == len(set(titles)), (
    "Duplicate contract titles detected."
)


--- TITLE CHECK ---
Total titles: 408
Unique titles: 408
Duplicate titles: 0


In [ ]:
qa_ids = []

for contract in contracts:
    for paragraph in contract["paragraphs"]:
        for qa in paragraph["qas"]:
            qa_ids.append(qa["id"])

print("\n--- QA ID CHECK ---")
print("QA IDs:", len(qa_ids))
print("Unique QA IDs:", len(set(qa_ids)))

assert len(qa_ids) == len(set(qa_ids)), (
    "Duplicate QA IDs detected."
)


--- QA ID CHECK ---
QA IDs: 22450
Unique QA IDs: 22450


In [ ]:
span_errors = []

for contract in contracts:

    title = contract["title"]

    for paragraph in contract["paragraphs"]:

        context = paragraph["context"]

        for qa in paragraph["qas"]:

            for answer in qa["answers"]:

                start = answer["answer_start"]
                expected_text = answer["text"]
                end = start + len(expected_text)

                # Check boundaries
                if start < 0 or end > len(context):

                    span_errors.append({
                        "contract": title,
                        "qa_id": qa["id"],
                        "problem": "out_of_bounds",
                        "start": start,
                        "end": end,
                        "context_length": len(context)
                    })

                    continue

                actual_text = context[start:end]

                # Check exact match
                if actual_text != expected_text:

                    span_errors.append({
                        "contract": title,
                        "qa_id": qa["id"],
                        "problem": "text_mismatch",
                        "start": start,
                        "expected": expected_text,
                        "actual": actual_text
                    })

In [ ]:
print("\n--- SPAN INTEGRITY ---")
print("Invalid spans:", len(span_errors))

if span_errors:
    print(span_errors[:5])

assert len(span_errors) == 0, (
    f"Found {len(span_errors)} invalid spans."
)


--- SPAN INTEGRITY ---
Invalid spans: 0


In [ ]:
span_records = []

for contract_index, contract in enumerate(contracts):

    for paragraph_index, paragraph in enumerate(
        contract["paragraphs"]
    ):

        for qa in paragraph["qas"]:

            for answer in qa["answers"]:

                start = answer["answer_start"]
                text = answer["text"]
                end = start + len(text)

                span_records.append({
                    "contract_index": contract_index,
                    "title": contract["title"],
                    "paragraph_index": paragraph_index,
                    "qa_id": qa["id"],
                    "question": qa["question"],
                    "start": start,
                    "end": end,
                    "text": text
                })

spans_df = pd.DataFrame(span_records)

In [ ]:
duplicate_span_mask = spans_df.duplicated(
    subset=[
        "contract_index",
        "paragraph_index",
        "question",
        "start",
        "end",
        "text"
    ],
    keep=False
)

duplicate_spans = spans_df[
    duplicate_span_mask
].copy()

print("\n--- EXACT DUPLICATE SPANS ---")
print("Duplicate span records:", len(duplicate_spans))


--- EXACT DUPLICATE SPANS ---
Duplicate span records: 0


In [ ]:
overlapping_spans = []

for (contract_index, paragraph_index), group in spans_df.groupby(
    ["contract_index", "paragraph_index"]
):

    records = group.to_dict("records")

    for i in range(len(records)):

        for j in range(i + 1, len(records)):

            a = records[i]
            b = records[j]

            overlaps = (
                a["start"] < b["end"]
                and b["start"] < a["end"]
            )

            if overlaps:

                overlapping_spans.append({
                    "title": a["title"],
                    "span_1_question": a["question"],
                    "span_1_start": a["start"],
                    "span_1_end": a["end"],
                    "span_2_question": b["question"],
                    "span_2_start": b["start"],
                    "span_2_end": b["end"]
                })

In [ ]:
print("\n--- OVERLAPPING / NESTED SPANS ---")
print("Overlap pairs:", len(overlapping_spans))


--- OVERLAPPING / NESTED SPANS ---
Overlap pairs: 2009


In [ ]:
def get_contract_text(contract):

    return "\n".join(
        paragraph["context"]
        for paragraph in contract["paragraphs"]
    )

In [ ]:
def get_context_group_id(contract):

    text = get_contract_text(contract)

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

In [ ]:
context_groups = {}

for index, contract in enumerate(contracts):

    group_id = get_context_group_id(contract)

    context_groups.setdefault(
        group_id,
        []
    ).append(index)

In [ ]:
duplicate_context_groups = {
    group_id: indices
    for group_id, indices in context_groups.items()
    if len(indices) > 1
}

print("\n--- DUPLICATE CONTRACT TEXT ---")

print(
    "Duplicate-text groups:",
    len(duplicate_context_groups)
)

for group_id, indices in duplicate_context_groups.items():

    print("\nContext group:", group_id[:12])

    for index in indices:
        print(" -", contracts[index]["title"])


--- DUPLICATE CONTRACT TEXT ---
Duplicate-text groups: 0


In [ ]:
category_to_index = {
    category: index
    for index, category in enumerate(categories)
}

In [ ]:
contract_labels = np.zeros(
    (len(contracts), len(categories)),
    dtype=int
)

for contract_index, contract in enumerate(contracts):

    for paragraph in contract["paragraphs"]:

        for qa in paragraph["qas"]:

            # Positive category if there is at least one span
            if len(qa["answers"]) > 0:

                category_index = category_to_index[
                    qa["question"]
                ]

                contract_labels[
                    contract_index,
                    category_index
                ] = 1

In [ ]:
print("Category matrix shape:", contract_labels.shape)

assert contract_labels.shape == (408, 41)

Category matrix shape: (408, 41)


In [ ]:
group_ids = list(context_groups.keys())

group_labels = []

for group_id in group_ids:

    contract_indices = context_groups[group_id]

    combined_presence = np.max(
        contract_labels[contract_indices],
        axis=0
    )

    group_labels.append(combined_presence)

group_labels = np.array(group_labels)

print(
    "Number of context groups:",
    len(group_ids)
)

print(
    "Group label matrix:",
    group_labels.shape
)

Number of context groups: 407
Group label matrix: (407, 41)


In [88]:
splitter =  MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_SEED
)

X = np.zeros(
    (len(group_ids), 1)
)

train_group_indices, validation_group_indices = next(
    splitter.split(
        X,
        group_labels
    )
)

In [89]:
train_group_ids = {
    group_ids[index]
    for index in train_group_indices
}

validation_group_ids = {
    group_ids[index]
    for index in validation_group_indices
}

In [90]:
train_contracts = []
validation_contracts = []

for contract in contracts:

    group_id = get_context_group_id(contract)

    if group_id in train_group_ids:

        train_contracts.append(contract)

    elif group_id in validation_group_ids:

        validation_contracts.append(contract)

    else:

        raise ValueError(
            f"No split assignment for "
            f"{contract['title']}"
        )

In [91]:
context_overlap = (
    train_group_ids
    & validation_group_ids
)

assert len(context_overlap) == 0, (
    "Context-group leakage detected!"
)

In [92]:
train_titles = {
    contract["title"]
    for contract in train_contracts
}

validation_titles = {
    contract["title"]
    for contract in validation_contracts
}

contract_overlap = (
    train_titles
    & validation_titles
)

assert len(contract_overlap) == 0, (
    "Contract leakage detected!"
)

In [93]:
assert (
    len(train_contracts)
    + len(validation_contracts)
    == 408
), "Contracts were lost during splitting."

In [94]:
print("\n--- FINAL SPLIT ---")

print("Seed:", RANDOM_SEED)

print(
    "Training contracts:",
    len(train_contracts)
)

print(
    "Validation contracts:",
    len(validation_contracts)
)

print(
    "Training percentage:",
    round(
        len(train_contracts) / 408 * 100,
        2
    )
)

print(
    "Validation percentage:",
    round(
        len(validation_contracts) / 408 * 100,
        2
    )
)

print(
    "Contract overlap:",
    len(contract_overlap)
)

print(
    "Context-group overlap:",
    len(context_overlap)
)


--- FINAL SPLIT ---
Seed: 20260901
Training contracts: 326
Validation contracts: 82
Training percentage: 79.9
Validation percentage: 20.1
Contract overlap: 0
Context-group overlap: 0


In [95]:
def get_category_support(contract_list):

    support = {
        category: 0
        for category in categories
    }

    for contract in contract_list:

        present = set()

        for paragraph in contract["paragraphs"]:

            for qa in paragraph["qas"]:

                if len(qa["answers"]) > 0:
                    present.add(qa["question"])

        for category in present:
            support[category] += 1

    return support

In [96]:
train_support = get_category_support(
    train_contracts
)

validation_support = get_category_support(
    validation_contracts
)

category_support_df = pd.DataFrame({
    "category": categories,

    "train_contracts": [
        train_support[c]
        for c in categories
    ],

    "validation_contracts": [
        validation_support[c]
        for c in categories
    ]
})

category_support_df

,category,train_contracts,validation_contracts
0,Highlight the parts (if any) of this contract ...,38,9
1,Highlight the parts (if any) of this contract ...,14,3
2,Highlight the parts (if any) of this contract ...,301,76
3,Highlight the parts (if any) of this contract ...,235,67
4,Highlight the parts (if any) of this contract ...,139,37
5,Highlight the parts (if any) of this contract ...,181,50
6,Highlight the parts (if any) of this contract ...,76,19
7,Highlight the parts (if any) of this contract ...,48,12
8,Highlight the parts (if any) of this contract ...,61,15
9,Highlight the parts (if any) of this contract ...,326,82


In [97]:
train_document_lengths = [
    len(get_contract_text(contract))
    for contract in train_contracts
]

document_length_stats = pd.Series(
    train_document_lengths
).describe()

print("\n--- TRAINING DOCUMENT LENGTH ---")
print(document_length_stats)


--- TRAINING DOCUMENT LENGTH ---
count       326.000000
mean      53878.389571
std       56011.375368
min        1081.000000
25%       15739.750000
50%       35374.000000
75%       68911.250000
max      338211.000000
dtype: float64


In [98]:
density_records = []

for contract in train_contracts:

    span_count = sum(
        len(qa["answers"])
        for paragraph in contract["paragraphs"]
        for qa in paragraph["qas"]
    )

    density_records.append({
        "title": contract["title"],
        "document_characters": len(
            get_contract_text(contract)
        ),
        "span_count": span_count
    })

annotation_density_df = pd.DataFrame(
    density_records
)

annotation_density_df.head()

,title,document_characters,span_count
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,54290,46
1,NELNETINC_04_08_2020-EX-1-JOINT FILING AGREEMENT,1081,4
2,ADAMSGOLFINC_03_21_2005-EX-10.17-ENDORSEMENT A...,24632,23
3,"KIROMICBIOPHARMA,INC_05_11_2020-EX-10.23-CONSU...",18403,24
4,"VEONEER,INC_02_21_2020-EX-10.11-JOINT VENTURE ...",8257,16


In [99]:
train_indices = [
    contracts.index(contract)
    for contract in train_contracts
]

train_label_matrix = contract_labels[
    train_indices
]

cooccurrence_matrix = (
    train_label_matrix.T
    @ train_label_matrix
)

cooccurrence_df = pd.DataFrame(
    cooccurrence_matrix,
    index=categories,
    columns=categories
)

cooccurrence_df

,"Highlight the parts (if any) of this contract related to ""Affiliate License-Licensee"" that should be reviewed by a lawyer. Details: Does the contract contain a license grant to a licensee (incl. sublicensor) and the affiliates of such licensee/sublicensor?","Highlight the parts (if any) of this contract related to ""Affiliate License-Licensor"" that should be reviewed by a lawyer. Details: Does the contract contain a license grant by affiliates of the licensor or that includes intellectual property of affiliates of the licensor?","Highlight the parts (if any) of this contract related to ""Agreement Date"" that should be reviewed by a lawyer. Details: The date of the contract","Highlight the parts (if any) of this contract related to ""Anti-Assignment"" that should be reviewed by a lawyer. Details: Is consent or notice required of a party if the contract is assigned to a third party?","Highlight the parts (if any) of this contract related to ""Audit Rights"" that should be reviewed by a lawyer. Details: Does a party have the right to audit the books, records, or physical locations of the counterparty to ensure compliance with the contract?","Highlight the parts (if any) of this contract related to ""Cap On Liability"" that should be reviewed by a lawyer. Details: Does the contract include a cap on liability upon the breach of a party’s obligation? This includes time limitation for the counterparty to bring claims or maximum amount for recovery.","Highlight the parts (if any) of this contract related to ""Change Of Control"" that should be reviewed by a lawyer. Details: Does one party have the right to terminate or is consent or notice required of the counterparty if such party undergoes a change of control, such as a merger, stock sale, transfer of all or substantially all of its assets or business, or assignment by operation of law?","Highlight the parts (if any) of this contract related to ""Competitive Restriction Exception"" that should be reviewed by a lawyer. Details: This category includes the exceptions or carveouts to Non-Compete, Exclusivity and No-Solicit of Customers above.","Highlight the parts (if any) of this contract related to ""Covenant Not To Sue"" that should be reviewed by a lawyer. Details: Is a party restricted from contesting the validity of the counterparty’s ownership of intellectual property or otherwise bringing a claim against the counterparty for matters unrelated to the contract?","Highlight the parts (if any) of this contract related to ""Document Name"" that should be reviewed by a lawyer. Details: The name of the contract",...,"Highlight the parts (if any) of this contract related to ""Renewal Term"" that should be reviewed by a lawyer. Details: What is the renewal term after the initial term expires? This includes automatic extensions and unilateral extensions with prior notice.","Highlight the parts (if any) of this contract related to ""Revenue/Profit Sharing"" that should be reviewed by a lawyer. Details: Is one party required to share revenue or profit with the counterparty for any technology, goods, or services?","Highlight the parts (if any) of this contract related to ""Rofr/Rofo/Rofn"" that should be reviewed by a lawyer. Details: Is there a clause granting one party a right of first refusal, right of first offer or right of first negotiation to purchase, license, market, or distribute equity interest, technology, assets, products or services?","Highlight the parts (if any) of this contract related to ""Source Code Escrow"" that should be reviewed by a lawyer. Details: Is one party required to deposit its source code into escrow with a third party, which can be released to the counterparty upon the occurrence of certain events (bankruptcy, insolvency, etc.)?","Highlight the parts (if any) of this contract related to ""Termination For Convenience"" that should be reviewed by a lawyer. Details: Can a party terminate this contract without cause (solely by giving a notice and allowing 

In [103]:
manifest_records = []

for contract in contracts:

    group_id = get_context_group_id(contract)

    if group_id in train_group_ids:
        split = "train"
    else:
        split = "validation"

    manifest_records.append({
        "title": contract["title"],
        "context_group_id": group_id,
        "split": split
    })

manifest_df = pd.DataFrame(
    manifest_records
)

manifest_df.to_csv(
    MANIFEST_PATH,
    index=False
)

In [110]:
train_data = {
    "version": data["version"],
    "data": train_contracts
}

validation_data = {
    "version": data["version"],
    "data": validation_contracts
}

In [111]:
with open(
    TRAIN_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        train_data,
        file,
        indent=2
    )


with open(
    VALIDATION_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        validation_data,
        file,
        indent=2
    )

In [112]:
evidence = {
    "seed": RANDOM_SEED,

    "source_file": DATA_PATH.name,

    "counts": {
        "contracts": num_contracts,
        "documents": num_documents,
        "categories": len(categories),
        "annotation_sets": num_annotation_sets,
        "spans": num_spans
    },

    "integrity": {
        "invalid_spans": len(span_errors),
        "duplicate_span_records": len(
            duplicate_spans
        ),
        "duplicate_context_groups": len(
            duplicate_context_groups
        )
    },

    "split": {
        "train_contracts": len(
            train_contracts
        ),
        "validation_contracts": len(
            validation_contracts
        ),
        "contract_overlap": len(
            contract_overlap
        ),
        "context_group_overlap": len(
            context_overlap
        )
    }
}

In [113]:
EVIDENCE_PATH = (
    OUTPUT_DIR
    / "split_verification.json"
)

with open(
    EVIDENCE_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        evidence,
        file,
        indent=2
    )

# CUAD Data Validation and Train/Validation Split

## Overview

This document describes the process used to validate the training-approved
CUAD dataset and create the fixed 80/20 training and validation split.

The goal was to ensure that the dataset was valid, prevent data leakage,
preserve duplicate-context relationships, and create a reproducible split
for future modeling.

---

## Source Data

Source file:

`data/cuad/train_separate_questions.json`

The training-approved dataset contains 408 contracts.

The original CUAD dataset was not used directly for the train/validation
split because the project provides a separate training-approved dataset.

---

## Expected Dataset Totals

The dataset was checked against the following required totals:

| Metric | Expected |
|---|---:|
| Contracts | 408 |
| Documents | 408 |
| Categories | 41 |
| Annotation Sets | 16,728 |
| Spans | 11,180 |

---

## Annotation Set Validation

The dataset contains 22,450 raw QA records.

However, these raw QA records are not equivalent to Annotation Sets.
Some contracts contain multiple QA records for the same category.

Each of the 408 contracts contains exactly 41 unique categories.

Therefore, Annotation Sets are counted using unique:

`Contract × Category`

combinations.

408 contracts × 41 categories = 16,728 Annotation Sets.

The repeated QA records were preserved rather than deleted because they
may contain separate annotation information.

---

## Data Integrity Checks

The following integrity checks were performed:

- Verified the expected number of contracts.
- Verified the expected number of documents.
- Verified all 41 categories are represented.
- Verified the expected number of Annotation Sets.
- Verified the expected number of Spans.
- Checked required IDs for uniqueness.
- Checked that annotated Spans fall within their source document.
- Checked that Span text exactly matches the referenced source text.
- Identified exact duplicate Spans without automatically deleting them.
- Preserved overlapping and nested Spans.

---

## Duplicate Contract Handling

Contracts with identical text were not deleted.

A `context_group_id` was generated using a SHA-256 hash of the contract
text.

Contracts with identical text therefore receive the same
`context_group_id`.

This ensures duplicate-context contracts remain together during the
train/validation split and cannot leak across datasets.

---

## Train/Validation Split

The split uses:

- Training-approved data only
- 80% training / 20% validation target
- Random seed: `20260901`
- Iterative multilabel stratification
- `context_group_id` as the split unit
- Combined 41-category presence for each context group

The split is performed at the context-group level rather than directly
at the contract level.

This means contracts sharing identical context cannot appear in different
splits.

---

## Leakage Checks

After creating the split, the following checks were performed:

- No Contract appears in both training and validation.
- No `context_group_id` appears in both training and validation.
- All 408 training-approved Contracts are assigned to exactly one split.

The final split is frozen and should not be regenerated using different
seeds based on model performance.

---

## Training Data Profile

The training-data analysis includes:

- Document-length statistics
- Category support
- Category co-occurrence
- Annotation density
- Duplicate-context analysis
- Span integrity information
- Data needed to compare future chunking strategies

---

## Generated Files

The workflow produces:

`train.json`
- Frozen training dataset.

`validation.json`
- Frozen validation dataset.

`split_manifest.csv`
- Records each Contract, its `context_group_id`, and its assigned split.

`split_verification.json`
- Machine-readable evidence containing dataset counts, integrity checks,
  seed information, and split-isolation results.

---

## Reproducibility

The split uses the fixed random seed:

`20260901`

The same approved input data, configuration, and code should reproduce
the same split.

The split should not be changed because another random split produces
better model performance.

---

## Remaining Work

The following items still need to be completed or independently verified:

- Complete the anomaly log.
- Record severity, owner, status, impact, and final decision for anomalies.
- Generate final training-profile tables and figures.
- Confirm official-test information is protected from normal development output.
- Have someone other than me  independently verify record
  counts, split isolation, leakage protections, and bundle integrity.